# IMPORT LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import pytz 
import openpyxl
import os 
from openpyxl import load_workbook
from openpyxl.styles import numbers

# DATA PREPARATION

In [ ]:
# Define the local time zone and get the current local time
local_tz = pytz.timezone('Asia/Jakarta')
now_local = datetime.now(local_tz)

# Define the desired date format and format the current local time
date_format = "%d %B %Y"
date_str = now_local.strftime(date_format) # Convert the datetime object to a string in the specified format

In [ ]:
# Import the files module from google.colab for file uploading
from google.colab import files
file = files.upload()

In [ ]:
# Get the filename from the uploaded file dictionary
uploaded_filename = list(file.keys())[0]

df = pd.read_csv(uploaded_filename)
df_backup = df.copy(deep=True)

# DATA CLEANING

In [ ]:
# Rename the 'shipping_number' column to 'shipping_number (Box #)' for clarity
df = df.rename(columns={'shipping_number': 'shipping_number (Box #)'})

# Define the desired order and subset of columns for the DataFrame
df_columns = ['driver_name', 'driver_phone', 'district', 'customer_name', 'customer_email',
              'delivery_date', 'address', 'address_note', 'customer_phone', 'order_no',
              'customer_status', 'paid_status', 'packaging_option', 'distance_in_km',
              'Zone_Name', 'Zone_Price', 'hubs', 'total_price', 'external_note',
              'internal_note', 'customer_note', 'time_slot', 'no_plastic',
              'payment_method', 'latitude', 'longitude', 'total_weight_perorder', 'shipping_number (Box #)']

# Create a deep copy of the original DataFrame as a backup
df_backup = df.copy(deep=True)

# Create a new DataFrame 'df' containing only the specified columns and make a copy to avoid SettingWithCopyWarning
df = df[df_columns].copy()

# Convert 'driver_name' to uppercase and remove 'MITRA-' or 'mitra-' prefix
df.loc[:, 'driver_name'] = df['driver_name'].str.upper().str.replace('^(MITRA-|mitra-)', '', regex=True)
# Replace 'ALL' in 'packaging_option' with 'BIGBOX + BIGBOX'
df.loc[:, 'packaging_option'] = df['packaging_option'].str.replace('^(ALL)', 'BIGBOX + BIGBOX', regex=True)
# Round 'distance_in_km' to 2 decimal places
df.loc[:, 'distance_in_km'] = df['distance_in_km'].round(2)
# Replace specific time slot variations with standardized values
df.loc[:, 'time_slot'] = df['time_slot'].str.replace('^(slot-12bb)', 'slot-1bb', regex=True).str.replace('^(slot-b2b-2)', 'slot-0bb', regex=True)

# Sort the DataFrame by 'hubs', 'driver_name', and 'customer_name'
df = df.sort_values(by=['hubs', 'driver_name', 'customer_name'], ascending=[True, True, True])

In [ ]:
df.head()

# DATA MANIPULATION

## STYLING B2B AND CASH ON DELIVERY ORDERS

In [ ]:
# Define a function to apply conditional highlighting and borders to rows
def highlight_b2b_and_payment(row):
  """
  Applies conditional highlighting and borders to a DataFrame row.

  - Rows with 'time_slot' as 'slot-0bb' or 'slot-1bb' will have a light blue background.
  - Rows with 'payment_method' as 'Cash on Delivery' will have green text.
  - All cells will have a light gray border.

  Args:
    row (pd.Series): A row from a DataFrame.

  Returns:
    list: A list of style strings for each cell in the row.
  """
  # Get the values from the 'time_slot' and 'payment_method' columns for the current row
  time_slot_value = row.loc['time_slot']
  payment_method_value = row.loc['payment_method']

  # Initialize a list to store styles for each cell in the row
  styles = []
  # Iterate through each item (cell value) in the current row
  for item in row:
    # Define default background and text colors, and border style
    background_color = '#FFFFFF'  # Default background color (white)
    text_color = '#000000'  # Default text color (black)
    border_style = '0.5px solid #D3D3D3' # Define border style

    # Apply light blue background if the time slot is 'slot-0bb' or 'slot-1bb'
    if time_slot_value in ('slot-0bb', 'slot-1bb'):
      background_color = '#7393B3'  # Light blue color

    # Apply green text color if the payment method is 'Cash on Delivery'
    # Using 'if' here allows this style to be applied in addition to the background color if both conditions are met
    if payment_method_value == 'Cash on Delivery':
      text_color = 'green'  # Green text color

    # Append the combined style string for the current cell to the styles list
    styles.append(f'background-color: {background_color}; color: {text_color}; border: {border_style}')

  # Return the list of style strings for the row
  return styles

## COLUMNS LEN ADJUSTMENT

In [ ]:
# Define a function to adjust column width and center align text in an Excel file
def adjust_excel_columns(filename):
    """
    Loads an Excel workbook, center aligns text in all cells of the header row,
    applies left alignment to columns A and C from the second row onwards,
    auto-adjusts column widths for specific columns, and saves the modified workbook.

    Args:
        filename (str): The path to the Excel file.
    """
    try:
        # Load the workbook
        workbook = openpyxl.load_workbook(filename)

        # Iterate through all sheets in the workbook
        for sheet_name in workbook.sheetnames:
            sheet = workbook[sheet_name]

            # Iterate through rows and columns to set alignment and date format
            for row_index, row in enumerate(sheet.iter_rows()):
                for cell in row:
                    if row_index == 0:  # Apply center alignment to the header row
                        cell.alignment = openpyxl.styles.Alignment(horizontal='center', vertical='center')
                    elif cell.column_letter in ('A', 'C'):  # Apply left alignment to columns A and C from the second row
                        cell.alignment = openpyxl.styles.Alignment(horizontal='left', vertical='center')

                    # Find the column index for 'delivery_date' if the header row is processed
                    if row_index == 0 and cell.value == 'delivery_date':
                      delivery_date_col_idx = cell.column - 1 # Get the 0-based index of the delivery_date column


            # Apply date format to the 'delivery_date' column from the second row onwards
            for row_index in range(1, sheet.max_row): # Start from the second row (index 1)
              cell = sheet.cell(row=row_index + 1, column=delivery_date_col_idx + 1) # Get the cell in the delivery_date column
              cell.number_format = 'YYYY-MM-DD' # Apply the date format

            # Auto-adjust column width for specific columns in the current sheet
            for column in sheet.columns:
                column_letter = openpyxl.utils.get_column_letter(column[0].column)
                # Apply width adjustment only to columns A and C
                if column_letter in ('A', 'C'):
                    max_length = 0
                    for cell in column:
                        try:
                            if len(str(cell.value)) > max_length:
                                max_length = len(str(cell.value))
                        except:
                            pass
                    adjusted_width = (max_length + 1.5)
                    sheet.column_dimensions[column_letter].width = adjusted_width


        # Save the modified workbook
        workbook.save(filename)

        print(f"Text in '{filename}' has been aligned and columns adjusted for all sheets.")

        # Optionally download the modified file
        files.download(filename)

    except FileNotFoundError:
        print(f"Error: The file '{filename}' was not found.")
    except Exception as e:
        print(f"An error occurred while processing '{filename}': {e}")

# EXPORT TO EXCEL FILES

In [ ]:
def export_delivery_list_to_excel(time_slot_filter_regex, slot_name, sheet_suffix):
    # Filter df to include only rows where 'time_slot' matches the given regex
    filter_by_time_slot = df[df['time_slot'].str.contains(time_slot_filter_regex, case=True)]

    # Group the filtered DataFrame by the 'hubs' column
    groups = filter_by_time_slot.groupby('hubs')

    # Create an Excel writer object with a dynamic filename based on the current date and slot
    filename = f"List Delivery {date_str} {slot_name}.xlsx"
    writer = pd.ExcelWriter(filename)

    # Iterate through each group (each unique 'hubs' value)
    for hubs, group_df in groups:
        # Define the sheet name for the current group
        sheet_name = f"{hubs}{sheet_suffix}"
        # Apply conditional formatting and borders to the current group's DataFrame
        formatted_df = group_df.style \
        .apply(highlight_b2b_and_payment, axis=1) \
        .set_table_styles([
          {'selector': '', 'props': [('border', '0.5px solid #D3D3D3')]}
      ])
        # Write the formatted group DataFrame to a sheet in the Excel file
        # index=False prevents writing the DataFrame index to the Excel file
        formatted_df.to_excel(writer, sheet_name=sheet_name, index=False)

    # Save the Excel file by closing the writer object
    writer.close()

    # Call the function to adjust columns after creating the Excel file
    adjust_excel_columns(filename)

    # Print a success message indicating the filename of the exported file
    print(f"{filename} is exported successfully")

In [ ]:
# Call the function for SLOT-0
export_delivery_list_to_excel('slot-0|slot-0bb|slot-b2b-2', 'SLOT-0', '0')

In [ ]:
# Call the function for SLOT-1
export_delivery_list_to_excel('slot-1|slot-1bb|slot-12bb', 'SLOT-1', '1')